<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 06 — K-Nearest Neighbors (KNN)
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Classificando por Distância no Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>
<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📏 Classificação por Distância</span>
</div>


## Chegou a hora do terceiro modelo

Nas Aulas 04 e 05 você treinou dois modelos baseados em **equações**: a Regressão
Linear (previu um número) e a Regressão Logística (previu uma probabilidade).
Hoje o raciocínio muda completamente.

O **K-Nearest Neighbors (KNN)** não calcula nenhuma fórmula complexa. A ideia é
simples — tão simples que você já usa isso no dia a dia:

> *"Para classificar algo desconhecido, olhe para os exemplos mais parecidos que
> você já viu."*

Você faz isso quando recomenda um restaurante ("você gostou daquele? então vai
gostar desse também") ou quando decide se vai gostar de um filme comparando com
outros parecidos que já assistiu.

### O problema de hoje

Vamos usar o KNN para prever se um passageiro do Titanic **sobreviveu ou não**
(voltamos a classificar, como na Aula 05 — mas agora sem nenhuma equação por trás).

### Roteiro de hoje

| Parte | Tema |
|-------|------|
| **1** | Intuição do KNN — votação de vizinhos |
| **2** | Distância — a régua que o KNN usa para saber quem é "parecido" |
| **3** | Treinando o KNN no Titanic |
| **4** | Por que normalizar é obrigatório no KNN |
| **5** | Escolhendo o K ideal e avaliando o modelo |

> **Tempo estimado: 35 minutos**

<div style="background:#d4edda; border-left:5px solid #155724; padding:14px 20px; border-radius:6px; margin:12px 0;">
<strong style="color:#155724;">A mesma interface, um novo jeito de pensar.</strong>
<span style="color:#155724;"> Você já usou <code>modelo.fit()</code> e <code>modelo.predict()</code> nas Aulas 04 e 05. O KNN usa exatamente a mesma interface — mas por dentro, a lógica é totalmente diferente: em vez de calcular uma equação, ele memoriza os dados e compara distâncias.</span>
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.model_selection import train_test_split

# ── Carregando e preparando o Titanic (mesma limpeza das aulas anteriores) ────
df = sns.load_dataset("titanic").copy()

df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

df["tamanho_familia"] = df["sibsp"] + df["parch"] + 1
df["sex_enc"]         = (df["sex"] == "female").astype(int)

# 5 features simples e fáceis de explicar
FEATURES = ["pclass", "sex_enc", "age", "tamanho_familia", "fare"]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("✅ Dataset pronto!")
print(f"   Treino: {len(X_treino)} passageiros  |  Teste: {len(X_teste)} passageiros")
print(f"   Features usadas: {FEATURES}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Intuição do KNN — Classificar pelo que é Parecido</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Diga-me com quem andas e te direi quem és."</p>
    </div>
</div>

### A ideia central

O KNN faz uma coisa simples:

> Para classificar um novo ponto, encontre os **K exemplos mais parecidos** no
> conjunto de treino e faça uma **votação**: a classe mais votada é a previsão.

Imagine que chega um passageiro novo e você precisa prever se ele sobreviveu.
O KNN busca os K passageiros mais parecidos (por idade, classe, gênero...) e
pergunta: *"Dos K mais parecidos, quantos sobreviveram?"*

**Duas coisas importantes sobre o KNN:**

| Característica | O que significa na prática |
|----------------|------------------------------|
| **Não existe "treino" de verdade** | O modelo só guarda os dados na memória. Todo o trabalho acontece na hora de prever. |
| **Tudo depende de "parecido"** | "Parecido" precisa virar um número — a **distância**. Veremos isso já na Parte 2. |

<div style="background:#e3f2fd; border-left:5px solid #1565c0; padding:12px 18px; border-radius:6px; margin:12px 0;">
<strong style="color:#1565c0;">Antes de rodar qualquer código:</strong>
<span style="color:#1565c0;"> a Missão 1 abaixo é um exercício manual de votação — sem Python. Fazer isso à mão uma vez cria uma compreensão que nenhuma biblioteca vai dar.</span>
</div>


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Antes de qualquer código, olhe para a tabela abaixo e responda: se K=3, qual classe o KNN prevê para o Passageiro Novo? E se K=5?</span></div>

### Exercício manual — Votação KNN

O **Passageiro Novo** tem: Idade=28, Classe=1ª, Gênero=Feminino.

Os 5 vizinhos mais próximos encontrados são:

| Vizinho | Distância | Idade | Classe | Gênero | Sobreviveu? |
|---------|-----------|-------|--------|--------|-------------|
| V1 | 0.8 | 26 | 1ª | Feminino | ✅ Sim |
| V2 | 1.1 | 30 | 1ª | Feminino | ✅ Sim |
| V3 | 1.4 | 25 | 2ª | Feminino | ✅ Sim |
| V4 | 1.9 | 32 | 1ª | Masculino | ❌ Não |
| V5 | 2.3 | 27 | 3ª | Feminino | ✅ Sim |

*✏️ Com K=3: o modelo analisa V1, V2 e V3. Votos: ___ Sim, ___ Não. Previsão: ___*

*✏️ Com K=5: o modelo analisa todos. Votos: ___ Sim, ___ Não. Previsão: ___*

*✏️ Esse passageiro novo realmente sobreviveu? Com base nos dados do Titanic, faz sentido?*


In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# print("Gabarito:")
# print()
# print("K=3: analisa V1, V2, V3")
# print("  Votos: 3 Sim, 0 Não → Previsão: SOBREVIVEU")
# print()
# print("K=5: analisa V1, V2, V3, V4, V5")
# print("  Votos: 4 Sim, 1 Não → Previsão: SOBREVIVEU")
# print()
# print("Faz sentido? SIM!")
# print("Mulher, 1ª classe, jovem → taxa de sobrevivência histórica muito alta")
# print("O KNN captura exatamente esse padrão nos dados.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Distância — A Régua que o KNN Usa</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Parecido" é subjetivo para humanos. Para o computador, precisa ser um número.</p>
    </div>
</div>

### Como medir "parecido" com um número?

A resposta mais comum é a **distância Euclidiana** — a mesma fórmula de
distância entre dois pontos que você já viu em geometria (Teorema de Pitágoras):

```
d(A, B) = √( (A₁−B₁)² + (A₂−B₂)² )        ← com 2 características
```

Vamos ver isso primeiro no jeito mais simples possível: **2 passageiros, 2
características** (idade e tarifa), desenhados num plano — exatamente como
calcular a distância entre dois pontos num mapa.


In [ ]:
# Distância entre 2 passageiros usando só 2 características: idade e tarifa
idade_A, tarifa_A = 25, 50
idade_B, tarifa_B = 30, 80

dist = np.sqrt((idade_A - idade_B)**2 + (tarifa_A - tarifa_B)**2)

plt.figure(figsize=(6, 6))
plt.scatter([idade_A], [tarifa_A], color="#0f3460", s=150, zorder=5, label="Passageiro A")
plt.scatter([idade_B], [tarifa_B], color="#e94560", s=150, zorder=5, label="Passageiro B")
plt.plot([idade_A, idade_B], [tarifa_A, tarifa_B], color="#f0a500",
         linestyle="--", linewidth=2, label=f"Distância = {dist:.1f}")

plt.xlabel("Idade"); plt.ylabel("Tarifa (£)")
plt.title("Distância Euclidiana entre 2 passageiros", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

print(f"d(A, B) = √(({idade_A}-{idade_B})² + ({tarifa_A}-{tarifa_B})²)")
print(f"        = √({(idade_A-idade_B)**2} + {(tarifa_A-tarifa_B)**2})")
print(f"        = √{(idade_A-idade_B)**2 + (tarifa_A-tarifa_B)**2}")
print(f"        = {dist:.2f}")


A mesma ideia funciona com **3, 4 ou 100 características** — só não dá mais
para desenhar num papel. A fórmula geral simplesmente soma o quadrado da
diferença em cada característica:

```
d(A, B) = √( (A₁−B₁)² + (A₂−B₂)² + ... + (Aₙ−Bₙ)² )
```

Vamos calcular isso para 2 passageiros reais, agora com 4 características.


In [ ]:
# Calculando distância euclidiana com mais características (passo a passo)
# Passageiro descrito por: [classe, sexo (1=mulher), idade normalizada, tam. família]
p_ana   = np.array([2.0, 1.0, 0.3, 1.0])
p_bruno = np.array([0.0, 0.0, 1.2, 3.0])

diff  = p_ana - p_bruno
diff2 = diff ** 2
soma  = diff2.sum()
dist  = np.sqrt(soma)

print("Ana:   ", p_ana)
print("Bruno: ", p_bruno)
print()
print(f"Diferenças:          {diff}")
print(f"Diferenças ao quadrado: {diff2}")
print(f"Soma dos quadrados:  {soma:.4f}")
print(f"Raiz quadrada:       {dist:.4f}  ← distância final")
print()
print(f"✅ Confere com np.linalg.norm(): {np.linalg.norm(p_ana - p_bruno):.4f}")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Calcule a distância entre o Passageiro Referência e os Passageiros C e D abaixo usando <code>np.linalg.norm()</code>. Qual dos dois está mais perto da Referência?</span></div>

In [ ]:
# ✏️ MISSÃO 2 — complete o código
p_ref = np.array([1.5, 1.0, 0.5, 1.0])   # Passageiro Referência
p_C   = np.array([1.2, 1.0, 0.3, 1.0])   # Passageiro C
p_D   = np.array([0.0, 0.0, 1.8, 4.0])   # Passageiro D

# ✏️ Calcule as duas distâncias com np.linalg.norm()
# dist_C = ???
# dist_D = ???

# print(f"Distância Ref → C: {dist_C:.4f}")
# print(f"Distância Ref → D: {dist_D:.4f}")
# print("O Passageiro mais próximo da Referência é: ???")


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# dist_C = np.linalg.norm(p_ref - p_C)
# dist_D = np.linalg.norm(p_ref - p_D)
#
# print(f"Distância Ref → C: {dist_C:.4f}")
# print(f"Distância Ref → D: {dist_D:.4f}")
# print(f"Passageiro C está mais próximo (dist={dist_C:.4f} < {dist_D:.4f})")
# print()
# print("C e Referência têm classe, sexo e idade parecidos — faz sentido!")


### Do cálculo de distância à votação

Agora que sabemos medir "parecido", o algoritmo do KNN é só juntar as peças
que já vimos — em 4 passos simples:

```
1. Calcular a distância do novo ponto até TODOS os pontos de treino
2. Ordenar do mais próximo ao mais distante
3. Pegar os K primeiros (os "K vizinhos mais próximos")
4. Fazer a votação — a classe mais comum entre os K vizinhos vence
```

Vamos implementar isso, um passo de cada vez, para um **passageiro novo e
imaginário**: idade 29.37 e tarifa £61.42 — valores bem específicos, de
propósito, para garantir que ele não seja idêntico a ninguém do treino.


In [ ]:
# Passo 1 e 2: calcular a distância do novo passageiro até todos os vizinhos e ordenar
# (usaremos só 2 características aqui para manter simples: idade e tarifa)
X_2feat     = X_treino[["age", "fare"]].values
y_arr       = y_treino.values
x_novo      = np.array([29.37, 61.42])   # passageiro novo e imaginário

distancias = []
for i in range(len(X_2feat)):
    dist = np.linalg.norm(x_novo - X_2feat[i])
    distancias.append((dist, y_arr[i]))

distancias.sort(key=lambda par: par[0])   # ordena do mais próximo ao mais distante

print("5 vizinhos mais próximos do novo passageiro:")
print(f"  {'#':>3} {'Distância':>10} {'Sobreviveu?'}")
for i, (dist, classe) in enumerate(distancias[:5], 1):
    print(f"  {i:>3} {dist:>10.4f} {'✅ Sim' if classe == 1 else '❌ Não'}")


In [ ]:
# Passo 3 e 4: pegar os K vizinhos mais próximos e fazer a votação
k = 5
k_vizinhos = distancias[:k]

votos = {}
for dist, classe in k_vizinhos:
    votos[classe] = votos.get(classe, 0) + 1

previsao = max(votos, key=votos.get)

print(f"Votação entre os {k} vizinhos mais próximos: {votos}")
print(f"Previsão do KNN para o passageiro imaginário: "
      f"{'Sobreviveu ✅' if previsao == 1 else 'Não sobreviveu ❌'}")


Isso é o KNN por completo — sem nenhuma mágica escondida. Na prática usamos o
`scikit-learn`, que faz exatamente esses 4 passos de forma otimizada. Antes de
ir para lá, veja como o valor de K muda o comportamento do modelo — o gráfico
abaixo usa um exemplo 2D simples (não é o Titanic) só para deixar isso bem visível.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification

# Dataset 2D simples, só para visualizar a fronteira de decisão
np.random.seed(42)
X_vis, y_vis = make_classification(
    n_samples=120, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1,
    class_sep=1.0, random_state=42
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("KNN — Fronteira de Decisão para K=1, K=5 e K=15", fontsize=13, fontweight="bold")

Ks = [1, 5, 15]
h  = 0.04

for ax, k in zip(axes, Ks):
    knn_vis = KNeighborsClassifier(n_neighbors=k)
    knn_vis.fit(X_vis, y_vis)

    x_min, x_max = X_vis[:,0].min()-0.5, X_vis[:,0].max()+0.5
    y_min, y_max = X_vis[:,1].min()-0.5, X_vis[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = knn_vis.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25, cmap="RdBu")
    ax.contour(xx, yy, Z, colors="gray", linewidths=0.5, alpha=0.5)
    ax.scatter(X_vis[:,0], X_vis[:,1], c=y_vis, cmap="RdBu",
               edgecolors="white", linewidth=0.6, s=55, zorder=5)

    score = knn_vis.score(X_vis, y_vis)
    ax.set_title(f"K = {k}   (acurácia treino: {score:.0%})", fontweight="bold")
    ax.set_xlabel("Feature 1"); ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.show()

print("Observe:")
print("  K=1  → fronteira muito irregular, decora cada ponto (overfitting)")
print("  K=5  → fronteira suave e razoável")
print("  K=15 → fronteira muito simples, ignora detalhes (underfitting)")
print()
print("Voltaremos a essa ideia (overfitting/underfitting) na Parte 5,")
print("agora aplicada aos dados reais do Titanic.")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Treinando o KNN no Titanic</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Mesma interface das Aulas 04 e 05 — fit() e predict()."</p>
    </div>
</div>

### Treinando o modelo

Vamos prever `survived` usando as mesmas 5 características simples que já
carregamos no início da aula: classe, sexo, idade, tamanho da família e tarifa
paga. A interface é idêntica à que você já usou nas Aulas 04 e 05: criar,
`fit()`, `predict()`.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# 1. Criar o modelo
modelo_knn = KNeighborsClassifier(n_neighbors=5)

# 2. Treinar — o KNN apenas memoriza os dados de treino
modelo_knn.fit(X_treino, y_treino)

# 3. Prever — só agora ele calcula as distâncias, uma por uma
y_previsto = modelo_knn.predict(X_teste)

acertos = (y_previsto == y_teste.values).sum()
total   = len(y_teste)

print("🎉 Primeiro modelo KNN treinado!")
print(f"   Acertos no teste: {acertos} de {total}  →  {acertos/total:.1%} de acurácia")


<div style="background:#e8d5f5; border-left:5px solid #5b2c8d; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#5b2c8d;">💡 </strong><span style="color:#5b2c8d;">Note que o <code>fit()</code> foi instantâneo — o modelo só memorizou os dados. O trabalho de verdade aconteceu no <code>predict()</code>, quando ele calculou as distâncias para cada passageiro novo. Isso é o que chamamos de <strong>Lazy Learning</strong> (aprendizado preguiçoso).</span></div>

In [ ]:
# Visualizando acertos e erros no conjunto de teste
fig, ax = plt.subplots(figsize=(10, 4))

cores = ["#e94560" if real != prev else "#0f3460"
         for real, prev in zip(y_teste.values, y_previsto)]

ax.scatter(range(len(y_teste)), y_teste.values, c=cores, s=30, alpha=0.7, zorder=5)

n_erros   = sum(r != p for r, p in zip(y_teste.values, y_previsto))
n_acertos = len(y_teste) - n_erros

ax.set_xlabel("Índice do Passageiro no Teste")
ax.set_ylabel("Sobreviveu (1=Sim, 0=Não)")
ax.set_title(f"Acertos e Erros no Conjunto de Teste (K=5 | Acertos: {n_acertos} | Erros: {n_erros})",
             fontweight="bold")

patch_acerto = mpatches.Patch(color="#0f3460", label=f"Acerto ({n_acertos})")
patch_erro   = mpatches.Patch(color="#e94560", label=f"Erro ({n_erros})")
ax.legend(handles=[patch_acerto, patch_erro], loc="upper right")

plt.tight_layout()
plt.show()


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Efeito da Escala — Por que Normalização é Obrigatória</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Sem normalização, o KNN ignora variáveis pequenas sem perceber."</p>
    </div>
</div>

### O problema concreto

O KNN mede distâncias. Se `fare` vai de 0 a mais de 500 e `sex_enc` vai de 0 a
1, qualquer cálculo de distância vai ser **dominado pela tarifa** — não porque
ela é mais importante, mas porque seus números são muito maiores.

É como comparar distâncias em metros e em centímetros sem converter: os
centímetros vão parecer enormes perto dos metros.


In [ ]:
# Visualizando o problema de escala nas features do Titanic
print("Escalas das features ANTES da normalização:")
print(f"  {'Feature':<18} {'Mínimo':>8} {'Máximo':>9}")
print("  " + "-"*38)
for col in FEATURES:
    print(f"  {col:<18} {X_treino[col].min():>8.2f} {X_treino[col].max():>9.2f}")

print()
print("⚠️  'fare' chega a mais de 500, enquanto 'sex_enc' vai só de 0 a 1.")
print("    Sem normalização, a tarifa sozinha decide quem é 'parecido'.")


In [ ]:
# Comparação direta: KNN COM vs SEM normalização
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)
X_teste_sc  = scaler.transform(X_teste)

# Sem normalização (dados brutos)
knn_sem_norm = KNeighborsClassifier(n_neighbors=5)
knn_sem_norm.fit(X_treino, y_treino)
acc_sem = knn_sem_norm.score(X_teste, y_teste)

# Com normalização (StandardScaler)
knn_com_norm = KNeighborsClassifier(n_neighbors=5)
knn_com_norm.fit(X_treino_sc, y_treino)
acc_com = knn_com_norm.score(X_teste_sc, y_teste)

print("COMPARAÇÃO: Com vs Sem Normalização (K=5)")
print(f"  Sem normalização:  acurácia = {acc_sem:.1%}")
print(f"  Com normalização:  acurácia = {acc_com:.1%}")
print(f"  Diferença: {(acc_com - acc_sem)*100:+.1f} pontos percentuais")

fig, ax = plt.subplots(figsize=(7, 4))
barras = ax.bar(["Sem Normalização", "Com Normalização"],
                [acc_sem * 100, acc_com * 100],
                color=["#e94560", "#0f3460"], edgecolor="white", width=0.5)
for b, v in zip(barras, [acc_sem*100, acc_com*100]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5, f"{v:.1f}%",
            ha="center", fontsize=13, fontweight="bold")
ax.set_ylabel("Acurácia (%)")
ax.set_title("Impacto da Normalização no KNN (K=5)", fontweight="bold")
plt.tight_layout()
plt.show()


<div style="background:#f8d7da; border-left:5px solid #721c24; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#721c24;">🚨 </strong><span style="color:#721c24;">Este gráfico é a demonstração mais importante desta aula: a mesma implementação do KNN, com os mesmos dados, produz resultados diferentes dependendo da normalização. <strong>Para o KNN, normalizar não é opcional — é parte do algoritmo.</strong></span></div>

---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;">
    <div>
        <span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span>
        <h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Escolhendo o K Ideal e Avaliando o Modelo</h2>
        <p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"K pequeno decora, K grande simplifica demais."</p>
    </div>
</div>

### O dilema do K

Vimos na Parte 2 que K=1 decora os dados (overfitting) e K muito grande ignora
padrões (underfitting). No Titanic real, como escolher o K certo? Testando
vários valores e comparando a acurácia no treino e no teste.

**Regra prática para começar:** `K_inicial ≈ √(número de amostras de treino)`,
e de preferência **ímpar**, para evitar empates na votação.


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Complete o loop abaixo para testar K de 1 a 30 e plotar a acurácia no treino e no teste. Identifique: (a) onde ocorre overfitting, (b) onde ocorre underfitting, (c) qual K você escolheria.</span></div>

In [ ]:
# ✏️ MISSÃO 3 — complete o código
Ks               = range(1, 31)
acuracias_treino = []
acuracias_teste  = []

for k in Ks:
    # ✏️ crie um KNeighborsClassifier(n_neighbors=k), treine com X_treino_sc,
    #    e guarde a acurácia de treino e teste (dica: use o método .score())
    # knn_k = ???
    # knn_k.fit(???, ???)
    # acuracias_treino.append(???)
    # acuracias_teste.append(???)
    pass

# ✏️ Depois, plote Ks x acuracias_treino e Ks x acuracias_teste


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# Ks               = range(1, 31)
# acuracias_treino = []
# acuracias_teste  = []
# for k in Ks:
#     knn_k = KNeighborsClassifier(n_neighbors=k)
#     knn_k.fit(X_treino_sc, y_treino)
#     acuracias_treino.append(knn_k.score(X_treino_sc, y_treino))
#     acuracias_teste.append(knn_k.score(X_teste_sc, y_teste))
#
# melhor_k = list(Ks)[np.argmax(acuracias_teste)]
# plt.plot(Ks, [a*100 for a in acuracias_treino], "o--", label="Treino")
# plt.plot(Ks, [a*100 for a in acuracias_teste], "s-", label="Teste")
# plt.axvline(melhor_k, linestyle="--", color="gray", label=f"Melhor K={melhor_k}")
# plt.xlabel("K"); plt.ylabel("Acurácia (%)"); plt.legend(); plt.show()


In [ ]:
# Executando para continuar a aula (mesmo que você não tenha completado a Missão 3)
Ks               = range(1, 31)
acuracias_treino = []
acuracias_teste  = []

for k in Ks:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_treino_sc, y_treino)
    acuracias_treino.append(knn_k.score(X_treino_sc, y_treino))
    acuracias_teste.append(knn_k.score(X_teste_sc,   y_teste))

melhor_k   = list(Ks)[np.argmax(acuracias_teste)]
melhor_acc = max(acuracias_teste)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(Ks, [a*100 for a in acuracias_treino], "o--",
        color="#e94560", linewidth=2, markersize=5, label="Treino")
ax.plot(Ks, [a*100 for a in acuracias_teste],  "s-",
        color="#0f3460", linewidth=2, markersize=5, label="Teste")
ax.axvline(melhor_k, color="#f0a500", linestyle="--", linewidth=1.8,
           label=f"Melhor K = {melhor_k}  ({melhor_acc:.1%})")
ax.annotate("Overfitting\n(K pequeno)", xy=(2, ax.get_ylim()[0]+1), fontsize=9,
            color="#e94560", ha="center")
ax.annotate("Underfitting\n(K grande)", xy=(28, ax.get_ylim()[0]+1), fontsize=9,
            color="#0f3460", ha="center")
ax.set_xlabel("Valor de K"); ax.set_ylabel("Acurácia (%)")
ax.set_title("Acurácia por Valor de K — Treino vs Teste (Titanic, KNN)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(f"✅ Melhor K encontrado: {melhor_k}  →  Acurácia no teste: {melhor_acc:.1%}")
print(f"   Regra √n sugeria K = {int(np.sqrt(len(X_treino_sc)))}")


### Avaliando além da acurácia

Imagine que 62% dos passageiros não sobreviveram. Um modelo que **sempre
prevê "não sobreviveu"**, sem aprender nada, já teria 62% de acurácia — e seria
inútil. Por isso olhamos métricas por classe:

| Métrica | Pergunta que responde |
|---------|------------------------|
| **Precisão** | Dos que previ como "Sobreviveu", quantos realmente sobreviveram? |
| **Recall** | Dos que realmente sobreviveram, quantos eu identifiquei? |
| **F1-Score** | Um equilíbrio entre Precisão e Recall |

A **Matriz de Confusão** mostra todos os acertos e erros de uma vez:

```
                    PREVISTO
                  Não Sobrev.   Sobreviveu
REAL  Não Sobrev.      VN            FP
      Sobreviveu       FN            VP
```


In [ ]:
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score
)

knn_final = KNeighborsClassifier(n_neighbors=melhor_k)
knn_final.fit(X_treino_sc, y_treino)
y_pred_final = knn_final.predict(X_teste_sc)

print(f"AVALIAÇÃO COMPLETA — KNN com K={melhor_k}")
print(f"  Acurácia:  {accuracy_score(y_teste, y_pred_final):.1%}")
print(f"  Precisão:  {precision_score(y_teste, y_pred_final):.1%}")
print(f"  Recall:    {recall_score(y_teste, y_pred_final):.1%}")
print(f"  F1-Score:  {f1_score(y_teste, y_pred_final):.3f}")


In [ ]:
# Matriz de confusão
fig, ax = plt.subplots(figsize=(5, 5))
cm = confusion_matrix(y_teste, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Não Sobrev.", "Sobreviveu"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Matriz de Confusão — KNN (K={melhor_k})", fontweight="bold")
plt.tight_layout()
plt.show()

vp, fn = cm[1,1], cm[1,0]
fp, vn = cm[0,1], cm[0,0]
print(f"VP={vp}: sobreviventes identificados corretamente")
print(f"VN={vn}: não-sobreviventes identificados corretamente")
print(f"FP={fp}: previu 'sobreviveu' mas não sobreviveu")
print(f"FN={fn}: previu 'não sobreviveu' mas sobreviveu")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 (final) — Olhando a Matriz de Confusão: (a) quantos sobreviventes o modelo deixou de identificar (FN)? (b) Recall ou Precisão está mais baixo? O que isso significa? (c) Como passageiro real, você prefere um erro de FP ou de FN? Por quê?</span></div>

*✏️ (a) Falsos Negativos: `???`*

*✏️ (b) Está mais baixo: `???` — significa que: `???`*

*✏️ (c) Eu prefiro errar por `???` porque: `???`*


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# print(f"(a) Falsos Negativos: {fn} sobreviventes não identificados pelo modelo.")
# print()
# prec = precision_score(y_teste, y_pred_final)
# rec  = recall_score(y_teste, y_pred_final)
# print(f"(b) Precisão = {prec:.1%}  |  Recall = {rec:.1%}")
# if rec < prec:
#     print("    Recall mais baixo: o modelo perde alguns sobreviventes reais.")
# else:
#     print("    Precisão mais baixa: o modelo erra prevendo sobrevivência demais.")
# print()
# print("(c) Como passageiro, o pior erro é o FN (o modelo diz que você não vai")
# print("    sobreviver, quando na verdade sobreviveria). Prefeririamos um modelo")
# print("    com Recall mais alto, mesmo que a Precisão caia um pouco.")


**✏️ Minha reflexão sobre a aula:**

1. KNN em uma frase simples: *...*

2. O insight mais importante para mim hoje foi: *...*

3. Em que situação real eu usaria KNN: *...*
